
# Bulk ATAC-seq End-to-End Pipeline (Mouse GRCm38.84)

This notebook performs:

1. FASTQ detection and samplesheet generation
2. nf-core/atacseq execution
3. Consensus peak aggregation
4. Gene activity matrix construction
5. Deconvolution against scRNA reference


In [ ]:

BASE_DIR = "/home/nakagawa/datasets/SRR_spinalcord"

ATAC_DIR = f"{BASE_DIR}/bulkATAC"

RESULTS_DIR = f"{BASE_DIR}/results_atac"

SAMPLESHEET = f"{BASE_DIR}/atac_samplesheet.csv"

H5AD_REF = "/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad"


In [ ]:

import os
import glob
import pandas as pd

def create_atac_samplesheet(fastq_dir, output_csv):

    r1_files = sorted(
        glob.glob(os.path.join(fastq_dir, "*_1.fastq.gz"))
    )

    records = []

    for r1 in r1_files:

        sample = os.path.basename(r1).split("_1.fastq.gz")[0]

        r2 = os.path.join(
            fastq_dir,
            f"{sample}_2.fastq.gz"
        )

        if os.path.exists(r2):
            records.append([sample, r1, r2, 1])

    df = pd.DataFrame(
        records,
        columns=["sample", "fastq_1", "fastq_2", "replicate"]
    )

    df.to_csv(output_csv, index=False)

    print(df.head())

    return df

atac_sheet = create_atac_samplesheet(
    ATAC_DIR,
    SAMPLESHEET
)


In [ ]:
%%bash

cd /home/nakagawa/datasets/SRR_spinalcord

nextflow run nf-core/atacseq \
    -profile singularity \
    --input atac_samplesheet.csv \
    --outdir ./results_atac \
    --genome GRCm38.84 \
    --narrow_peak \
    -resume


In [ ]:

import os
import pandas as pd

counts_path = os.path.join(
    RESULTS_DIR,
    "bwa",
    "mergedLibrary",
    "macs",
    "narrowPeak",
    "consensus",
    "consensus_peaks.mLb.clN.featureCounts.txt"
)

anno_path = os.path.join(
    RESULTS_DIR,
    "bwa",
    "mergedLibrary",
    "macs",
    "narrowPeak",
    "consensus",
    "consensus_peaks.mLb.clN.annotatePeaks.txt"
)

counts = pd.read_csv(
    counts_path,
    sep="\t",
    skiprows=1
)

anno = pd.read_csv(
    anno_path,
    sep="\t"
)

anno = anno.rename(
    columns={
        anno.columns[0]: "PeakID",
        "Gene Name": "gene_name"
    }
)

counts = counts.rename(
    columns={"Geneid": "PeakID"}
)

merged = pd.merge(
    counts,
    anno[["PeakID", "gene_name"]],
    on="PeakID",
    how="inner"
)

merged = merged.dropna(subset=["gene_name"])

sample_cols = [
    c for c in merged.columns
    if c.startswith("SRR")
]

merged = merged[["gene_name"] + sample_cols]

gene_activity = merged.groupby("gene_name").sum()

output_csv = os.path.join(
    BASE_DIR,
    "bulkATAC_gene_activity.csv"
)

gene_activity.to_csv(output_csv)

print(f"Saved: {output_csv}")

gene_activity.head()


In [ ]:

import scanpy as sc
import numpy as np
from scipy.optimize import nnls
import matplotlib.pyplot as plt

bulk_atac = pd.read_csv(
    f"{BASE_DIR}/bulkATAC_gene_activity.csv",
    index_col=0
)

adata_ref = sc.read_h5ad(H5AD_REF)


In [ ]:

sc.pp.normalize_total(adata_ref, target_sum=1e4)
sc.pp.log1p(adata_ref)

cell_type_key = "cell_type"

sc.tl.rank_genes_groups(
    adata_ref,
    groupby=cell_type_key,
    method="wilcoxon"
)

top_n_markers = 50

markers = []

for cl in adata_ref.obs[cell_type_key].unique():
    genes = sc.get.rank_genes_groups_df(
        adata_ref,
        group=cl
    )["names"].head(top_n_markers)

    markers.extend(genes)

overlapping = list(
    set(markers)
    & set(bulk_atac.index)
    & set(adata_ref.var_names)
)

print(f"Matched genes: {len(overlapping)}")


In [ ]:

signature_matrix = pd.DataFrame(index=overlapping)

for cl in adata_ref.obs[cell_type_key].unique():

    subset = adata_ref[
        adata_ref.obs[cell_type_key] == cl
    ]

    mean_expr = np.ravel(
        subset[:, overlapping].X.mean(axis=0)
    )

    signature_matrix[cl] = mean_expr

signature_matrix.head()


In [ ]:

deconv = pd.DataFrame(
    index=bulk_atac.columns,
    columns=signature_matrix.columns
)

A = signature_matrix.loc[overlapping].values

for sample in bulk_atac.columns:

    b = bulk_atac.loc[overlapping, sample].values

    if b.sum() > 0:
        b = (b / b.sum()) * 1e4

    sol, _ = nnls(A, b)

    if sol.sum() > 0:
        sol = sol / sol.sum()

    deconv.loc[sample] = sol

deconv = deconv.astype(float)

deconv


In [ ]:

deconv.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

plt.title("Bulk ATAC-seq Deconvolution")
plt.ylabel("Cell Fraction")
plt.xlabel("Samples")

plt.tight_layout()
plt.show()
